<a href="https://colab.research.google.com/github/cafauzi13/ESG_SentimentAnalysis/blob/main/scripts/Augmentasi_LLM_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Augmentasi Data — ESG Sentiment Analysis
**Strategi anti-limit API Gemini Free Tier:**
- Hard cap: 12 request/menit (aman di bawah limit 15 RPM)
- Exponential backoff + jitter saat kena 429
- Checkpoint otomatis ke Google Drive setiap 5 artikel
- Resume otomatis jika session mati di tengah jalan
- Teks dipotong 600 karakter untuk hemat token
- Estimasi total waktu ditampilkan sebelum mulai

## 0. Install & Mount Drive

In [2]:
!pip install -q google-genai scikit-learn

from google.colab import drive, userdata
drive.mount('/content/drive')

# ============================================================
# SESUAIKAN PATH INI
# ============================================================
DRIVE_DIR        = '/content/drive/MyDrive/'
CHECKPOINT_PATH  = DRIVE_DIR + 'augmented_checkpoint.csv'
TRAIN_OUT_PATH   = DRIVE_DIR + 'train_set_augmented.csv'
TEST_OUT_PATH    = DRIVE_DIR + 'test_set_asli.csv'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## 1. Setup Gemini Client

In [3]:
from google import genai
from google.genai import types

# Simpan API key via: Colab sidebar → kunci (🔑) → tambah GEMINI_API_KEY
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

MODEL_NAME = 'gemini-2.0-flash-lite'  # paling murah & cepat untuk free tier

# Test koneksi
test_resp = client.models.generate_content(
    model=MODEL_NAME,
    contents='Balas hanya dengan kata OK'
)
print(f'✅ Gemini OK: {test_resp.text.strip()}')

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 54.282206301s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@

## 2. Load Data & Train/Test Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

GITHUB_URL = 'https://raw.githubusercontent.com/cafauzi13/ESG_SentimentAnalysis/main/data/IndoBERT.csv'
df = pd.read_csv(GITHUB_URL)
df = df.dropna(subset=['Isi Berita Clean', 'Sentiment', 'Tag'])
df['Isi Berita Clean'] = df['Isi Berita Clean'].astype(str)
print(f'Total data: {len(df)}')
print(f'\nDistribusi Sentimen:\n{df["Sentiment"].value_counts()}')

# Train/test split — test set TIDAK diaugmentasi
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Sentiment']
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# Simpan test set sekali saja ke Drive
df_test.to_csv(TEST_OUT_PATH, index=False)

print(f'\nTrain: {len(df_train)} | Test: {len(df_test)}')
print(f'\nDistribusi Train:\n{df_train["Sentiment"].value_counts()}')
print(f'✅ Test set disimpan: {TEST_OUT_PATH}')

## 3. Estimasi Waktu & Konfigurasi Rate Limit

In [ ]:
import math

# ============================================================
# KONFIGURASI RATE LIMIT — jangan ubah kecuali pakai paid tier
# ============================================================
RPM_LIMIT      = 12    # request per menit (free tier max = 15, kita pakai 12 untuk aman)
SLEEP_BASE     = 60 / RPM_LIMIT   # 5 detik antar request
MAX_CHARS      = 600   # potong teks input untuk hemat token
MAX_OUT_TOKENS = 600   # output token per request
MAX_RETRIES    = 5     # maksimal retry saat kena 429
CHECKPOINT_N   = 5     # simpan checkpoint setiap N artikel

# Hitung kebutuhan augmentasi
max_count  = df_train['Sentiment'].value_counts().max()
total_need = sum(
    max(0, max_count - cnt)
    for cnt in df_train['Sentiment'].value_counts()
)

est_minutes = math.ceil(total_need * SLEEP_BASE / 60)

print('=' * 50)
print('  ESTIMASI AUGMENTASI')
print('=' * 50)
print(f'  Target per kelas  : {max_count}')
print(f'  Total perlu dibuat: {total_need} artikel')
print(f'  Jeda antar request: {SLEEP_BASE:.1f} detik')
print(f'  Estimasi waktu    : ~{est_minutes} menit')
print('=' * 50)
print('\n⚠️  Jangan tutup tab Colab. Jika session mati,')
print('   re-run dari cell ini — checkpoint otomatis resume.')

## 4. Fungsi Augmentasi dengan Backoff

In [ ]:
import time
import random

def augment_artikel(text, sentiment, tag, attempt_num=0):
    """
    Generate 1 parafrase artikel.
    Return: string teks baru, atau None jika gagal setelah MAX_RETRIES.
    """
    text_input = str(text)[:MAX_CHARS]

    prompt = (
        f"Kamu adalah parafrase generator untuk dataset riset berita ESG Indonesia.\n"
        f"Artikel asli (sentimen: {sentiment}, kategori ESG: {tag}):\n"
        f"{text_input}\n\n"
        f"Tugas: Buat 1 parafrase artikel di atas. Ketentuan:\n"
        f"- Pertahankan sentimen {sentiment} secara konsisten\n"
        f"- Pertahankan konteks ESG kategori {tag}\n"
        f"- Gunakan gaya bahasa berita Indonesia yang natural dan formal\n"
        f"- Ubah struktur kalimat dan pilihan kata, bukan sekadar ganti sinonim\n"
        f"- HANYA output teks parafrase saja, tanpa penjelasan, tanpa label apapun"
    )

    for attempt in range(MAX_RETRIES):
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=MAX_OUT_TOKENS,
                    temperature=0.75,
                )
            )
            hasil = resp.text.strip()
            if len(hasil) >= 100:
                return hasil
            else:
                print(f'    ⚠️ Output terlalu pendek ({len(hasil)} chars), retry {attempt+1}...')
                time.sleep(SLEEP_BASE)

        except Exception as e:
            err_str = str(e).lower()

            if '429' in err_str or 'quota' in err_str or 'rate' in err_str:
                # Exponential backoff dengan jitter
                wait = (2 ** (attempt + 1)) + random.uniform(1, 5)
                print(f'    🚦 Rate limit (429). Tunggu {wait:.0f} detik... (attempt {attempt+1}/{MAX_RETRIES})')
                time.sleep(wait)

            elif 'resource' in err_str or '503' in err_str or '500' in err_str:
                wait = 15 + random.uniform(0, 10)
                print(f'    🔄 Server error. Tunggu {wait:.0f} detik...')
                time.sleep(wait)

            else:
                print(f'    ❌ Error tidak dikenal: {e}')
                time.sleep(SLEEP_BASE)

    print(f'    ❌ Gagal setelah {MAX_RETRIES} percobaan, skip artikel ini.')
    return None

## 5. Jalankan Augmentasi (dengan Resume Otomatis)

In [ ]:
# ============================================================
# RESUME: load checkpoint jika ada
# ============================================================
if os.path.exists(CHECKPOINT_PATH):
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    augmented_rows = df_checkpoint.to_dict('records')
    print(f'🔄 Resume dari checkpoint: {len(augmented_rows)} artikel sudah ada')
    print(f'   Distribusi checkpoint:\n{df_checkpoint["Sentiment"].value_counts().to_string()}')
else:
    augmented_rows = []
    print('🆕 Mulai augmentasi dari awal')

# ============================================================
# AUGMENTASI PER KELAS SENTIMEN
# ============================================================
max_count = df_train['Sentiment'].value_counts().max()

for sentiment in df_train['Sentiment'].unique():
    df_kelas  = df_train[df_train['Sentiment'] == sentiment]
    sudah_ada = sum(1 for r in augmented_rows if r.get('Sentiment') == sentiment)
    kekurangan = max_count - len(df_kelas) - sudah_ada

    if kekurangan <= 0:
        print(f'\n✅ Sentimen "{sentiment}": sudah cukup ({len(df_kelas)} asli + {sudah_ada} sintetis), skip.')
        continue

    print(f'\n{"="*55}')
    print(f'  Sentimen: "{sentiment}" | Perlu generate: {kekurangan} artikel')
    print(f'  (Asli: {len(df_kelas)}, sudah di-checkpoint: {sudah_ada})')
    print(f'{"="*55}')

    # Sampling dengan replace agar bisa generate lebih banyak dari data asli
    samples = df_kelas.sample(n=kekurangan, replace=True, random_state=42).reset_index(drop=True)
    berhasil, gagal = 0, 0

    for i, row in samples.iterrows():
        nomor = i + 1
        print(f'  [{nomor:3d}/{kekurangan}] Generating "{sentiment}"... ', end='', flush=True)

        teks_baru = augment_artikel(
            row['Isi Berita Clean'],
            row['Sentiment'],
            row['Tag']
        )

        if teks_baru:
            new_row = row.to_dict()
            new_row['Isi Berita Clean'] = teks_baru
            new_row['is_augmented'] = True
            augmented_rows.append(new_row)
            berhasil += 1
            print(f'✅ ({len(teks_baru)} chars)')
        else:
            gagal += 1
            print('⚠️ Skip')

        # Checkpoint ke Drive setiap CHECKPOINT_N artikel
        if len(augmented_rows) % CHECKPOINT_N == 0 and augmented_rows:
            pd.DataFrame(augmented_rows).to_csv(CHECKPOINT_PATH, index=False)
            print(f'  💾 Checkpoint disimpan ({len(augmented_rows)} artikel total)')

        # Jeda wajib antar request
        time.sleep(SLEEP_BASE)

    print(f'\n  Hasil "{sentiment}": {berhasil} berhasil, {gagal} gagal')

# Checkpoint final
if augmented_rows:
    pd.DataFrame(augmented_rows).to_csv(CHECKPOINT_PATH, index=False)
    print(f'\n💾 Checkpoint final disimpan: {len(augmented_rows)} artikel sintetis')

## 6. Gabungkan & Simpan Train Set Final

In [ ]:
if not augmented_rows:
    print('❌ Tidak ada artikel sintetis. Periksa API key dan koneksi.')
else:
    df_train['is_augmented'] = False
    df_aug = pd.DataFrame(augmented_rows)
    df_aug['is_augmented'] = True

    df_train_final = pd.concat([df_train, df_aug], ignore_index=True)
    df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

    df_train_final.to_csv(TRAIN_OUT_PATH, index=False)

    print('=' * 55)
    print('  TRAIN SET FINAL')
    print('=' * 55)
    print(f'  Total artikel    : {len(df_train_final)}')
    print(f'  Asli             : {df_train_final["is_augmented"].eq(False).sum()}')
    print(f'  Sintetis         : {df_train_final["is_augmented"].eq(True).sum()}')
    print(f'\n  Distribusi Sentimen:')
    print(df_train_final['Sentiment'].value_counts().to_string())
    print(f'\n  Distribusi Tag:')
    print(df_train_final['Tag'].value_counts().to_string())
    print('=' * 55)
    print(f'\n✅ Disimpan: {TRAIN_OUT_PATH}')
    print(f'✅ Test set : {TEST_OUT_PATH}')
    print('\n→ Lanjut ke klasifikasi.ipynb')

## 7. Verifikasi Kualitas Augmentasi (Opsional)
Cek beberapa sampel hasil augmentasi secara manual

In [ ]:
import textwrap

df_aug_check = df_train_final[df_train_final['is_augmented'] == True].sample(3, random_state=1)

for i, (_, row) in enumerate(df_aug_check.iterrows()):
    print(f'\n{"─"*55}')
    print(f'  Sampel #{i+1} | Sentimen: {row["Sentiment"]} | Tag: {row["Tag"]}')
    print(f'{"─"*55}')
    print(textwrap.fill(str(row['Isi Berita Clean'])[:400], width=70))
    print('  ...')